# Session 9: 姿勢制御 — カスケードの概念
# Session 9: Attitude Control — The Cascade Concept

## 目標 / Objective

内側ループ (Rate) と外側ループ (Angle) のカスケード構造を理解する。

Understand cascade structure with inner loop (Rate) and outer loop (Angle).

## この Notebook で学ぶこと / What You'll Learn

| トピック | 説明 |
|---------|------|
| カスケード制御 | 内側/外側ループの帯域分離 |
| 単ループ vs カスケード | 性能の違いを定量比較 |
| 帯域設計 | 内側ループを外側より速くする理由 |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from stampfly_edu.sim.cascade import (
    simulate_cascade_attitude,
    simulate_single_loop_attitude,
    compare_cascade_vs_single,
)

print("Ready! / 準備完了！")

## 1. カスケード制御の構造 / Cascade Control Structure

### なぜカスケードが必要か？

ドローンの角度制御は2段階の問題：

```
角度目標 → [角度PID] → 角速度目標 → [レートPID] → モーター → [機体] → 角速度 → [積分] → 角度
              ↑                         ↑                        │              │
              │ 外側ループ（遅い）        │ 内側ループ（速い）       │              │
              └─────────────────────────│────────────────────────│──────────────┘
                                        └────────────────────────┘
```

### 帯域分離の原則

| ループ | 帯域 | 制御周期 | 目的 |
|--------|------|---------|------|
| 内側（Rate） | 高い（~50 rad/s） | 400Hz | 角速度の素早い安定化 |
| 外側（Angle） | 低い（~5 rad/s） | 400Hz | 目標角度への追従 |

内側ループの帯域は外側の **3〜10倍** にする。

In [ ]:
# Compare cascade vs single-loop
# カスケード vs 単ループを比較
fig = compare_cascade_vs_single(setpoint_deg=10.0, duration=1.5)
plt.show()

## 2. カスケードのゲイン調整 / Cascade Gain Tuning

In [ ]:
# Effect of outer loop gain
# 外側ループゲインの効果
fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

for Kp_angle, color, ls in [(2.0, 'b', '-'), (5.0, 'r', '-'), (10.0, 'g', '-'), (20.0, 'purple', '--')]:
    result = simulate_cascade_attitude(Kp_angle=Kp_angle, setpoint_deg=10.0, duration=1.5)
    axes[0].plot(result['time'], result['angle'], color=color, linestyle=ls,
                label=f'Kp_angle={Kp_angle}')
    axes[1].plot(result['time'], result['rate'], color=color, linestyle=ls)

axes[0].plot(result['time'], result['angle_sp'], 'k:', linewidth=0.8, label='Setpoint')
axes[0].set_ylabel('Angle (deg) / 角度')
axes[0].set_title('Effect of Outer Loop Gain / 外側ループゲインの効果')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_ylabel('Rate (deg/s) / 角速度')
axes[1].set_xlabel('Time (s) / 時間')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. 考察課題 / Discussion Questions

1. **帯域分離が崩れたら**: 外側ループのゲインを内側より大きくしたらどうなるか？
2. **3段カスケード**: 位置制御まで入れると何段のカスケードになるか？各段の帯域は？
3. **実機との違い**: シミュレーションでは見えないが実機で重要になる要素は何か？

---

1. **Bandwidth Separation Failure**: What happens if outer loop gain exceeds inner loop?
2. **3-Stage Cascade**: How many stages for position control? What bandwidth for each?
3. **Sim vs Reality**: What factors matter on real hardware but not in simulation?

In [ ]:
# Your experiments here / ここで実験してみてください
